In [1]:
import cv2
import numpy as np
import time

# Load image
img = cv2.imread("image1.jpeg", 0)

if img is None:
    raise ValueError("Image not found.")

# -----------------------------
# 1D Gaussian Kernel
# -----------------------------
def gaussian_kernel_1D(kernel_size, sigma):
    k = kernel_size // 2
    x = np.arange(-k, k+1)
    kernel = np.exp(-(x**2) / (2 * sigma**2))
    kernel = kernel / np.sum(kernel)
    return kernel

# -----------------------------
# 2D Gaussian Kernel (for comparison)
# -----------------------------
def gaussian_kernel_2D(kernel_size, sigma):
    k = kernel_size // 2
    x = np.arange(-k, k+1)
    y = np.arange(-k, k+1)
    xx, yy = np.meshgrid(x, y)
    kernel = np.exp(-(xx**2 + yy**2) / (2 * sigma**2))
    kernel = kernel / np.sum(kernel)
    return kernel

# -----------------------------
# 2D Convolution
# -----------------------------
def convolution2D(image, kernel):
    k = kernel.shape[0] // 2
    padded = np.pad(image, ((k,k),(k,k)), mode='edge')
    output = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i:i+kernel.shape[0], j:j+kernel.shape[1]]
            output[i,j] = np.sum(region * kernel)

    return output

# -----------------------------
# Separable Convolution
# -----------------------------
def separable_convolution(image, kernel_1D):
    k = len(kernel_1D) // 2
    
    # Horizontal pass
    padded = np.pad(image, ((0,0),(k,k)), mode='edge')
    temp = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded[i, j:j+len(kernel_1D)]
            temp[i,j] = np.sum(region * kernel_1D)

    # Vertical pass
    padded2 = np.pad(temp, ((k,k),(0,0)), mode='edge')
    output = np.zeros_like(image, dtype=np.float32)

    for i in range(image.shape[0]):
        for j in range(image.shape[1]):
            region = padded2[i:i+len(kernel_1D), j]
            output[i,j] = np.sum(region * kernel_1D)

    return output

# Parameters
kernel_size = 7
sigma = 2.0

# Generate kernels
kernel_2D = gaussian_kernel_2D(kernel_size, sigma)
kernel_1D = gaussian_kernel_1D(kernel_size, sigma)

# -----------------------------
# Timing 2D Convolution
# -----------------------------
start = time.time()
output_2D = convolution2D(img, kernel_2D)
time_2D = time.time() - start

# -----------------------------
# Timing Separable Convolution
# -----------------------------
start = time.time()
output_sep = separable_convolution(img, kernel_1D)
time_sep = time.time() - start

# -----------------------------
# Verify Results
# -----------------------------
difference = np.mean(np.abs(output_2D - output_sep))

print("2D Convolution Time:", round(time_2D,5), "sec")
print("Separable Convolution Time:", round(time_sep,5), "sec")
print("Mean Difference:", round(difference,5))


2D Convolution Time: 0.42322 sec
Separable Convolution Time: 0.8326 sec
Mean Difference: 0.0
